#### Data Quality Validation & Governance

**Project:** Queens Supermarket Sales ETL Pipeline  
**Client Context:** Midroc Commerce (Queens Supermarket)  

**Objective:**  
Implement comprehensive data quality checks, validation rules, anomaly detection, and generate a quality report. 
This phase demonstrates production-grade data engineering practices including data profiling, validation, error logging, and basic data governance — key requirements for the Midroc Data Engineer role.

#### Imports

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
import os
import json
from dotenv import load_dotenv
import sqlalchemy as sa
from sqlalchemy import create_engine, text

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

load_dotenv()

pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported successfully!")

Libraries imported successfully!


#### Connect to Database

In [2]:
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'queens_supermarket_dw')

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}",
    pool_pre_ping=True
)

print(f"Successfully connected to database: {DB_NAME}")

Successfully connected to database: queens_supermarket_dw


#### Data Quality Rules

In [3]:
dq_config = {
    "fact_sales": {
        "row_count_expected_min": 40000,
        "critical_columns": ["invoice_id", "date_key", "branch_key", "total", "quantity"],
        "numeric_checks": {
            "quantity": {"min": 1},
            "unit_price": {"min": 0},
            "total": {"min": 10},
            "tax_15": {"min": 0},
            "gross_margin_percentage": {"min": 0, "max": 100}
        },
        "uniqueness": ["invoice_id"],
        "referential_integrity": {
            "date_key": "dim_date",
            "branch_key": "dim_branch",
            "product_key": "dim_product"
        }
    }
}

print("Data Quality configuration loaded.")

Data Quality configuration loaded.


#### Run Data Quality Checks



In [4]:
print(" Running Data Quality Validation...\n")
quality_results = {}

with engine.connect() as conn:
    
    # 1. Row Count Check
    row_count = conn.execute(text("SELECT COUNT(*) FROM fact_sales")).scalar()
    quality_results["total_records"] = row_count
    print(f"Total Records in fact_sales : {row_count:,}")
    
    # 2. Completeness Check (Null values)
    print("\nCompleteness Check (Null counts):")
    null_check = conn.execute(text("""
        SELECT 
            SUM(CASE WHEN invoice_id IS NULL THEN 1 ELSE 0 END) as null_invoice,
            SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) as null_date,
            SUM(CASE WHEN total IS NULL THEN 1 ELSE 0 END) as null_total,
            SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) as null_quantity
        FROM fact_sales
    """)).fetchone()
    
    print(f"Null invoice_id : {null_check[0]}")
    print(f"Null date_key   : {null_check[1]}")
    print(f"Null total      : {null_check[2]}")
    print(f"Null quantity   : {null_check[3]}")

    # 3. Validity Checks
    print("\nValidity Checks:")
    validity = conn.execute(text("""
        SELECT 
            SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) as invalid_quantity,
            SUM(CASE WHEN total <= 0 THEN 1 ELSE 0 END) as invalid_total,
            SUM(CASE WHEN gross_margin_percentage < 0 OR gross_margin_percentage > 100 THEN 1 ELSE 0 END) as invalid_margin
        FROM fact_sales
    """)).fetchone()
    
    print(f"Invalid quantity (<=0)       : {validity[0]}")
    print(f"Invalid total (<=0)          : {validity[1]}")
    print(f"Invalid gross margin         : {validity[2]}")

    # 4. Uniqueness Check
    duplicate_invoices = conn.execute(text("""
        SELECT COUNT(*) FROM (
            SELECT invoice_id FROM fact_sales 
            GROUP BY invoice_id HAVING COUNT(*) > 1
        ) t
    """)).scalar()
    print(f"Duplicate invoices           : {duplicate_invoices}")

print("\n Data Quality checks completed!")

 Running Data Quality Validation...

Total Records in fact_sales : 50,000

Completeness Check (Null counts):
Null invoice_id : 0
Null date_key   : 0
Null total      : 0
Null quantity   : 0

Validity Checks:
Invalid quantity (<=0)       : 0
Invalid total (<=0)          : 0
Invalid gross margin         : 0
Duplicate invoices           : 0

 Data Quality checks completed!


#### Generate Data Quality Report

In [5]:
report = {
    "report_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "project": "Queens Supermarket Sales ETL",
    "total_records": quality_results.get("total_records"),
    "quality_score": "High" if duplicate_invoices == 0 and validity[0] == 0 and validity[1] == 0 else "Medium",
    "issues_found": {
        "null_values": int(null_check[0] + null_check[1] + null_check[2] + null_check[3]),
        "invalid_records": int(validity[0] + validity[1] + validity[2]),
        "duplicates": int(duplicate_invoices)
    },
    "status": "PASSED" if (duplicate_invoices == 0 and validity[0] == 0 and validity[1] == 0) else "WARNING"
}

# Save report as JSON
report_path = '../docs/data_quality_report.json'
os.makedirs('../docs', exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(report, f, indent=4)

print(f" Data Quality Report saved to: {report_path}")
print(f"Overall Status: {report['status']}")

 Data Quality Report saved to: ../docs/data_quality_report.json
Overall Status: PASSED


#### Final Summary



In [6]:
print("="*70)
print("DATA QUALITY VALIDATION SUMMARY")
print("="*70)
print(f"Total Records          : {report['total_records']:,}")
print(f"Quality Status         : {report['status']}")
print(f"Issues Found           : {report['issues_found']}")
print(f"Report Generated On    : {report['report_date']}")
print("="*70)

if report['status'] == "PASSED":
    print("All critical data quality checks PASSED!")
else:
    print("Some quality issues detected. Review recommended.")

print("\n Data Quality Validation Completed!")

DATA QUALITY VALIDATION SUMMARY
Total Records          : 50,000
Quality Status         : PASSED
Issues Found           : {'null_values': 0, 'invalid_records': 0, 'duplicates': 0}
Report Generated On    : 2026-04-29 08:23:52
All critical data quality checks PASSED!

 Data Quality Validation Completed!
